# Question 22 - 6. Zigzag Conversion

The string `"PAYPALISHIRING"` is written in a zigzag pattern on a given number of rows like this:

    P   A   H   N
    A P L S I I G
    Y   I   R

And then read line by line: `"PAHNAPLSIIGYIR"`

Write the code that will take a string and make this conversion given a number of rows:

    string convert(string s, int numRows);

**Example 1:**

    Input: s = "PAYPALISHIRING", numRows = 3
    Output: "PAHNAPLSIIGYIR"

**Example 2:**

    Input: s = "PAYPALISHIRING", numRows = 4
    Output: "PINALSIGYAHRPI"
    Explanation:
    P     I    N
    A   L S  I G
    Y A   H R
    P     I

**Example 3:**

    Input: s = "A", numRows = 1
    Output: "A"

**Constraints:**

- `1 <= s.length <= 1000`
- `s` consists of English letters (lower-case and upper-case), `','` and `'.'`.
- `1 <= numRows <= 1000`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Simulation with row buckets (bounce the row index between top and bottom)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (2D grid) | O(numRows · n) | O(numRows · n) |
| 2️⃣ Optimized (row buckets) | O(n) | O(n) |
| ➕ Alternative (cycle formula) | O(n) | O(1) extra |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Characters go **down** the rows one at a time, then diagonally **up** to the top, then down again. At the end I read the rows top to bottom. The output only depends on which **row** each character ends up in. The columns are just for drawing."*

- `numRows == 1`? → No zigzag. Return `s` unchanged. **(Needs a special case, or the direction flip breaks.)**
- `numRows >= len(s)`? → Each character gets its own row, so the result equals `s`.
- Return a new string.

### Step 1 · Brute force

**Idea:** Literally draw it. Allocate a `numRows × n` grid, walk the zigzag placing characters (go down; at the bottom, go diagonally up-right; at the top, go down again), then read the grid row by row, skipping blanks.

🗣️ *"The most literal solution builds the 2D picture and reads it row by row. It's correct, but the grid is mostly empty: numRows × n cells for n characters. With 1000 × 1000 that's a million cells for a thousand letters. The columns don't affect the output, so I don't need to store them."*

In [1]:
class SolutionBrute:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s
        grid = [[""] * len(s) for _ in range(numRows)]
        row, col, going_down = 0, 0, True
        for ch in s:
            grid[row][col] = ch
            if going_down:
                if row == numRows - 1:      # hit bottom: start going diagonally up
                    going_down = False
                    row -= 1
                    col += 1
                else:
                    row += 1
            else:
                if row == 0:                # hit top: start going down
                    going_down = True
                    row += 1
                else:
                    row -= 1
                    col += 1
        return "".join(ch for r in grid for ch in r if ch)

### Step 2 · Optimize

**Bottleneck:** the grid wastes space on empty cells and on column positions that never affect the answer.

**Key insight:** only the **row** of each character matters, and within a row characters appear in their original order. So keep `numRows` string buckets, walk `s` once, append each character to the bucket for its current row, and bounce the row index:
- At row 0 → direction = +1 (down)
- At row numRows−1 → direction = −1 (up)

Then join the buckets.

🗣️ *"I only care which row each character lands in, so I keep one list per row. I walk the string, append each character to the current row, and flip direction when I hit the top or bottom. Then I concatenate the rows. That's O(n) time, and O(n) space for the buckets."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Full 2D grid | ❌ | O(numRows·n) space, mostly empty. Column tracking adds bugs. |
| **List of row buckets + bouncing index** | ✅ | Direct simulation without the empty cells. Easy to explain and get right. |
| Row strings with `+=` | ⚠️ | Works in CPython (often optimised), but lists + `join` are guaranteed linear. |
| **Cycle formula** (period `2·numRows − 2`) | ✅ (alternative) | Visits characters in output order directly, so O(1) extra space. More index arithmetic, and the "middle rows take two characters per cycle" rule is easy to slip on. |

**Cycle formula idea:** the pattern repeats every `cycle = 2·(numRows − 1)` characters. Row `r` takes index `k·cycle + r`, and, for middle rows only, also `k·cycle + cycle − r`.

In [2]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s
        rows = [[] for _ in range(numRows)]
        row, step = 0, 1
        for ch in s:
            rows[row].append(ch)
            if row == 0:
                step = 1                  # bounce off the top
            elif row == numRows - 1:
                step = -1                 # bounce off the bottom
            row += step
        return "".join("".join(r) for r in rows)


class SolutionCycle:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s
        n, cycle = len(s), 2 * (numRows - 1)
        out = []
        for r in range(numRows):
            for base in range(0, n, cycle):
                if base + r < n:
                    out.append(s[base + r])                      # vertical stroke
                if 0 < r < numRows - 1 and base + cycle - r < n:
                    out.append(s[base + cycle - r])              # diagonal stroke
        return "".join(out)

### Step 3 · Toy example walkthrough

`s = "PAYPALISHIRING"`, `numRows = 3`. Row sequence: 0,1,2,1,0,1,2,1,0,…

| ch | P | A | Y | P | A | L | I | S | H | I | R | I | N | G |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| row | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 |

Buckets:
- row 0: `P A H N`
- row 1: `A P L S I I G`
- row 2: `Y I R`

Joined: `PAHN` + `APLSIIG` + `YIR` = **`PAHNAPLSIIGYIR`** ✅

**Cycle view** (cycle = 2·(3−1) = 4): row 0 takes indices 0, 4, 8, 12 → P, A, H, N. Row 1 (a middle row) takes 1, **3**, 5, **7**, 9, **11**, 13 (the bold ones come from the diagonal: `base + 4 − 1`).

In [3]:
cases = [
    ("PAYPALISHIRING", 3, "PAHNAPLSIIGYIR"),
    ("PAYPALISHIRING", 4, "PINALSIGYAHRPI"),
    ("A", 1, "A"),
    ("AB", 1, "AB"),
    ("ABC", 5, "ABC"),          # more rows than characters
    ("ABCD", 2, "ACBD"),
]
for s, rows, expected in cases:
    for Impl in (SolutionBrute, Solution, SolutionCycle):
        assert Impl().convert(s, rows) == expected, (Impl.__name__, s, rows)
    print(s, rows, "->", expected)

import random, string
for _ in range(500):
    s = "".join(random.choices(string.ascii_letters, k=random.randint(1, 20)))
    r = random.randint(1, 7)
    assert Solution().convert(s, r) == SolutionBrute().convert(s, r) == SolutionCycle().convert(s, r)
print("All tests passed ✅")

PAYPALISHIRING 3 -> PAHNAPLSIIGYIR
PAYPALISHIRING 4 -> PINALSIGYAHRPI
A 1 -> A
AB 1 -> AB
ABC 5 -> ABC
ABCD 2 -> ACBD
All tests passed ✅


### Step 4 · Wrap up

| | Time | Extra space | Why |
|---|---|---|---|
| 2D grid | O(numRows · n) | O(numRows · n) | Allocates and scans every cell |
| **Row buckets** | **O(n)** | **O(n)** | Each character is appended once and joined once |
| Cycle formula | O(n) | O(1) (besides output) | Computes each output index directly |

**Complexity, explained:** in the bucket version, every character is appended to exactly one bucket (O(1) amortized), and the final join copies each character once → O(n). The buckets hold all n characters → O(n) space. The cycle version outputs each index exactly once by arithmetic, with no buckets.

**Edge cases:** `numRows == 1` (the direction would never flip, so return early), `numRows ≥ len(s)`, two rows (alternating).

🗣️ **30-second recap:** *"Only the row each character lands in matters. I keep one bucket per row, walk the string with a row pointer that bounces between top and bottom, append each character to its row, then concatenate the rows. That's O(n) time and space. Special-case one row. There's also an O(1)-extra-space version using the cycle length 2·(rows−1)."*